In [ ]:
# ルートに移動

%cd ..

In [ ]:
# ライブラリのインポート

import json
import os
import shutil

import numpy as np
from ultralytics import SAM
from ultralytics.utils.ops import ltwh2xyxy

In [ ]:
# ディレクトリの定義

corrected_dir = os.path.join("data", "corrected")
raw_dir = os.path.join("data", "raw")

In [ ]:
# モデルの読み込み

model_path = os.path.join("weights", "sam2.1_l.pt")
model = SAM(model_path)

In [ ]:
# セグメンテーションの実行

os.makedirs(os.path.join(corrected_dir, "images"), exist_ok=True)
os.makedirs(os.path.join(corrected_dir, "annotations"), exist_ok=True)

threshold = 0.5

annotation_path = os.path.join(raw_dir, "annotations", "train.json")
with open(annotation_path, "r") as f:
    data = json.load(f)

for image in data["images"]:
    id = image["id"]
    file_name = image["file_name"]
    image_path = os.path.join(raw_dir, "images", file_name)

    src = image_path
    dst = os.path.join(corrected_dir, "images", file_name)
    shutil.copy2(src, dst)

    annotations = [ann for ann in data["annotations"] if ann["image_id"] == id]

    boxes = [ann["bbox"] for ann in annotations]
    boxes = np.array(boxes)
    boxes = ltwh2xyxy(boxes)

    width = image["width"]
    height = image["height"]

    results = model(image_path, bboxes=boxes)

    masks = results[0].masks.data.cpu().numpy()

    for annotation, mask in zip(annotations, masks):
        x, y, w, h = [int(b) for b in annotation["bbox"]]
        xyxy = ltwh2xyxy(np.array([[x, y, w, h]], dtype=np.float32))
        x1, y1, x2, y2 = xyxy.astype(np.int32)[0]

        mask = mask > 0.5

        touch = mask[y1 : y1 + 2, x1:x2].any() and mask[y2 - 2 : y2, x1:x2].any()

        if not touch:
            rows, cols = np.where(mask[y1:y2, x1:x2])

            x3 = x1 + int(cols.min())
            y3 = y1 + int(rows.min())
            x4 = x1 + int(cols.max()) + 1
            y4 = y1 + int(rows.max()) + 1

            annotation["bbox"] = [float(x3), float(y3), float(x4 - x3), float(y4 - y3)]

            x1, y1, x2, y2 = x3, y3, x4, y4

        denominator = max(1, (x2 - x1) * 2)
        ratio = float(mask[y2 - 2 : y2, x1:x2].sum()) / denominator

        if ratio > threshold:
            x1, y1, x2, y2 = x1, y1, x2, min(height, y2 + 40)

            results = model(image_path, bboxes=[[x1, y1, x2, y2]])
            masks = results[0].masks.data.cpu().numpy()

            index = int(np.argmax(masks.reshape(masks.shape[0], -1).sum(axis=1)))
            mask = masks[index] > 0.5

            rows, cols = np.where(mask[y1:y2, x1:x2])

            x3 = x1 + int(cols.min())
            y3 = y1 + int(rows.min())
            x4 = x1 + int(cols.max()) + 1
            y4 = y1 + int(rows.max()) + 1

            annotation["bbox"] = [float(x3), float(y3), float(x4 - x3), float(y4 - y3)]

save_path = os.path.join(corrected_dir, "annotations", "train.json")
with open(save_path, "w") as f:
    json.dump(data, f, indent=2)